<a href="https://colab.research.google.com/github/gabrielolin/16886-Embodied-AI-Safety/blob/main/%5BS26_EAISafety%5D%5BHW2%5D_Data_driven_HJ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U "numpy<2.0"

In [ ]:
!git clone https://github.com/CMU-IntentLab/eais_hw2.git
!gdown 1xmZEntbsMMEYuCapJfYw72GihKgiGA97
!gdown 1F3cnbZCtUw2v6HgwooHl_a8W5gqIUTsZ
!unzip checkpoints.zip
!pip install configargparse==1.7

# IMPORTANT

Use GPU runtime!
Runtime -> Change Runtime Type -> T4 GPU

# A Brief Recap on Reachability
We define a failure set $F$ as the zero sub-level set of a margin function $l(x)$, that is $F := \{ x \; | \; l(x) < 0 \}$. We can find a safe control-invariant set as the zero super-level set of a value function that satisfies the following terminal-time Hamilton Jacobi variational inequality (HJ-VI):

\begin{aligned}
0 = \min\{D_t V(x,t) + \max_{u \in U} &\nabla V(x,t)^\top f(x,u), l(x) - V(x,T)\} \\
V(x,T) &= l(x)
\end{aligned}

Or, in discrete time, the fixed-point safety bellman equation:

\begin{equation}
V(x) = \min \{l(x), \max_{u \in U} V(f(x,u))\}
\end{equation}

This value function can be used to recover an optimal safety-preserving policy (see HW 1).





---

# Self-supervised Learning: DeepReach


In class we briefly went over DeepReach, a self-supervised learning method for computing a Hamilton-Jacobi Value functions.


The original implementation of DeepReach uses two loss functions for learning the value function:

1.   Boundary Condition Loss: $L_1 = ||V_\theta(x_i, t_i) - l(x_i)||\;\mathbb{1}[t_i = T]$
2.   HJ VI Loss: $L_2 = ||min\{D_t V_\theta(x_i, t_i) + \max_u \nabla V_\theta(x_i, t_i)^\top f(x_i,u), \; l(x_i) - V_\theta(x_t, t_i) \}||$

The first loss function allows us to supervise the value function at the terminal time using a function $l(x)$ that is assumed to be known. The second loss function uses the HJ variational inequality to implicitly supervise the value function through knowledge of its gradients, since we do not have a ground truth value function to use as supervision.

### **Q1.1: DeepReach Boundary Condition**

Why do we need to enforce the boundary condition loss? Can you think of a degenerate solution that would minimize $L_2$?
Hint: Section V in the original paper https://arxiv.org/pdf/2011.02082


**TODO: Your Markdown Solution Here**



As a reminder, a 3D Dubin's car is represented as a planar vehicle whose state is position and heading $x := (p_x, p_y, \theta)$ and has continuous-time dynamics described by this ordinary differential equation (ODE) below:

\begin{align}
\dot{p}_x &= v \cos(\theta) \\
\dot{p}_y &= v \sin(\theta) \\
\dot{\theta} &= \omega
\end{align}


The robot’s linear velocity is fixed $v = 1$ m/s and it controls its angular velocity, $u := \omega$ which is bounded, $\omega \in [-1.25, 1.25]$. In the following examples, the failure set $\mathit{F}$ represents an obstacle centered at $(0, 0)$ and contains any state within a 0.5 m radius circle in the $p_x$-$p_y$ plane irrespective of the heading angle.

### **Q1.2 Run Vanilla DeapReach**

The following code runs the original DeepReach implementation. Upload the final validation plot and comment on the quality of the learned solution. ESTIMATED TIME: 8-10 minutes

**TODO: Your Markdown Solution Here**


Validation Plot Location: runs/dubins3d_vanilla_run/training/checkpoints/BRS_validation_plot_epoch_2500.png

In [ ]:
!python eais_hw2/deepreach/run_experiment.py --mode train --experiment_class DeepReach --dynamics_class Dubins3D --experiment_name dubins3d_vanilla_run --minWith target --goalR 0.5 --velocity 1. --omega_max 1.25 --angle_alpha_factor 1.2 --set_mode avoid --deepreach_model vanilla --pretrain

One issue that has bottlenecked DeepReach is the need to learn the terminal value $l(x)$ in order to learn a non-degenerate value function. In practice, poorly balancing the two loss functions can lead to unstable learning, or even a "collapse" of the value function where it returns a constant value.

A recent improvement to DeepReach instead choses to learn a *residual* value function: $V(x,t) = l(x) + g_\theta(x, t)$. In particular, the authors design this residual as: $g_\theta(x,t) = (T-t) O_\theta(x,t)$.

### **Q1.3 Residual DeepReach**

Why should we include the factor $(T - t)$ in the residual? What would we have to learn if this factor didn't exist?

HINT: We are solving a *terminal time* variational inequality. What happens at $t=T$?

**TODO: Your Markdown Solution Here**

### **Q1.4 Run Residual DeepReach**

We will now run DeepReach using exact boundary constraints. The following code run's the original DeepReach implementation. Upload the final validation plot and comment on the quality of the learned solution compared to the validation plot learned previously.** ESTIMATED TIME: 8-10 minutes

**TODO: Your Markdown Solution Here**

Validation Plot Location: runs/dubins3d_exact_run/training/checkpoints/BRS_validation_plot_epoch_2500.png

In [ ]:
!python eais_hw2/deepreach/run_experiment.py --mode train --experiment_class DeepReach --dynamics_class Dubins3D --experiment_name dubins3d_exact_run --minWith target --goalR 0.5 --velocity 1. --omega_max 1.25 --angle_alpha_factor 1.2 --set_mode avoid --deepreach_model exact --pretrain

---

# Reinforcement Learning


We will now learn the safety value function using reinforcement learning.

In particular, we will use DDQN where the discrete action space of the robot is $u \in \{-\omega_{max}, \; 0, \; \omega_{max}\}$.

In the discrete time case, we need to solve for the fixed-point bellman equation:

\begin{equation}
V(x) = \min\{l(x), \max_u V(f(x,u))\}
\end{equation}

In [ ]:
!pip install ruamel-yaml==0.17.4
!apt-get install swig -y
!pip install box2d-py==2.3.8

### **Q2.1: Warmup**

Show that $\forall x, V(x) \leq l(x)$

**TODO: Your markdown solution here**

In class, we saw that we needed to add a time-discounting factor $\gamma$ to the fixed-point bellman equation in order to induce a contraction mapping.

This discounted safety bellman equation looks like:
\begin{equation}
V(x) = (1 - \gamma) l(x) + \gamma \min \{l(x), \max_u V(f(x,u)) \}
\end{equation}

### **Q2.2 Run Reachability RL**

Let's first run DDQN with a low discount factor of $\gamma = 0.95$.

**You need to implement the discounted reach-avoid Bellman equation:**

1. Open `eais_hw2/safety_rl/RARL/DDQNSingleAvoid.py`
2. Find the `TODO` comment around line 160 in the `update` method
3. Implement the discounted reach-avoid Bellman equation for normal states:
   - The equation is: $V(x) = (1 - \gamma) \cdot l(x) + \gamma \cdot \min\{l(x), \max_u V(f(x,u))\}$
   - Here, the variable `g_x` represent l(x) and `state_value_nxt` V(f(x, u)).
   - One line is sufficient.

After implementing the TODO, run the cell below to train the DDQN agent. Select a plotted checkpoint and comment on the quality of the learned value function (the ground truth value function is overlayed). Upload the plot to the markdown file.

ESTIMATED TIME: 10 minutes.

**TODO: Your markdown solution here**

Plot paths: logs/experiments/car-DDQN/priviliged_state_g0.95-toEnd/figure/X0000.png

In [ ]:
!python eais_hw2/safety_rl/sim_dubins_avoid.py

# ![Value function](logs/experiments/car-DDQN/priviliged_state_g0.95-toEnd/figure/20000.png)

### **Q2.3 Run Annealed Reachability RL**

Now, we will anneal the discount factor from 0.95 to 0.9999 over the course of training.

**TODO: Let's first run DDQN with a low discount factor of $\gamma = 0.95$ and anneal to 0.9999. Select a plotted checkpoint and comment on the quality of the learned value function (the ground truth value function is overlayed). How is this different from the previous result? Upload the plot to the markdown file.**

ESTIMATED TIME: 10 minutes.

**TODO: Your markdown solution here**

Plot paths: logs/experiments/car-DDQN/priviliged_state_annealing_g0.95-toEnd/figure/X0000.png

In [ ]:
!python eais_hw2/safety_rl/sim_dubins_avoid.py --annealing=True

# ![Value function](logs/experiments/car-DDQN/priviliged_state_annealing_g0.95-toEnd/figure/100000.png) ![Value function checkpoint](logs/experiments/car-DDQN/priviliged_state_g0.95-toEnd/figure/20000.png)

---

# Latent Safety with the World Model (RSSM)

We now take our first step into latent-space reachability. In this section, we provide a pretrained world model (RSSM, Recurrent State Space Model) along with a failure classifier.

The world model is trained using an image reconstruction objective on a dataset of 5,000 trajectories collected from random actions.

To save your time, the world model has already been trained for you :). This includes both the RSSM training and the latent safety margin function, which was trained on top of the latent representations. The failure set is defined as a circle centered at (0, 0) with a radius of 0.5.

If you would like to train the world model yourself using the 3D Dubins car environment, you can do so by first generating training data: `python generate_data_traj.py`. This will create random trajectories for training. Then, run: `python dreamerv3-torch/dreamer_offline.py`.

Run the following cell to load the pretrained world model.

In [ ]:
#@title Load World Model

import sys
import pathlib
import os
import collections
import numpy as np
import torch
import gym
import imageio
import ruamel.yaml as yaml

wm_root = pathlib.Path('eais_hw2/dreamerv3-torch').resolve()
sys.path.insert(0, str(wm_root.parent))
sys.path.insert(0, str(wm_root))
import tools
import models
to_np = lambda x: x.detach().cpu().numpy()

def recursive_update(base, update):
    for k, v in update.items():
        if isinstance(v, dict) and k in base:
            recursive_update(base[k], v)
        else:
            base[k] = v

config_path = pathlib.Path('eais_hw2/configs.yaml').resolve()
yaml_loader = yaml.YAML(typ='safe', pure=True)
configs = yaml_loader.load(config_path.read_text())
defaults = {}
recursive_update(defaults, configs['defaults'])
class Config:
    pass
config = Config()
for k, v in defaults.items():
    setattr(config, k, v)
config.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config.traindir = pathlib.Path(config.logdir).expanduser() / 'train_eps'
config.evaldir = pathlib.Path(config.logdir).expanduser() / 'eval_eps'
config.steps = int(config.steps) // getattr(config, 'action_repeat', 1)
config.num_actions = 3
bounds = np.array([[config.x_min, config.x_max], [config.y_min, config.y_max], [0, 2 * np.pi]])
low, high = bounds[:, 0], bounds[:, 1]
midpoint, interval = (low + high) / 2.0, high - low
image_size = config.size[0]
observation_space = gym.spaces.Dict({
    'state': gym.spaces.Box(np.float32(midpoint - interval/2), np.float32(midpoint + interval/2)),
    'obs_state': gym.spaces.Box(low=-1, high=1, shape=(2,), dtype=np.float32),
    'image': gym.spaces.Box(low=0, high=255, shape=(image_size, image_size, 3), dtype=np.uint8),
})
action_space = gym.spaces.Discrete(3)

class DummyLogger:
    step = 0
    def write(self, **kw): pass
    def scalar(self, *a): pass
    def image(self, *a): pass
    def video(self, *a): pass

logger = DummyLogger()
agent = models.WorldModel(observation_space, action_space, 0, config).to(config.device)

# Dynamically generate 6 trajectories using generate_data_traj (no dataset load)
sys.path.insert(0, str(pathlib.Path('.').resolve()))
from eais_hw2.generate_data_traj import gen_one_traj_img

x_min, x_max = config.x_min, config.x_max
y_min, y_max = config.y_min, config.y_max
u_max = config.turnRate
radius = config.obs_r
dt, v = config.dt, config.speed
dpi = config.size[0]

def traj_to_episode(state_obs, acs, state_gt, img_obs, dones):
    T = len(img_obs)
    one_hot = lambda ac: [1, 0, 0] if ac < 0 else ([0, 1, 0] if ac == 0 else [0, 0, 1])
    return {
        'image': [np.array(img) for img in img_obs],
        'state': [np.array(s, dtype=np.float32) for s in state_gt],
        'obs_state': [np.array([np.cos(state_obs[t]), np.sin(state_obs[t])], dtype=np.float32) for t in range(T)],
        'action': [np.array(one_hot(ac), dtype=np.uint8) for ac in acs],
        'is_first': [t == 0 for t in range(T)],
        'is_terminal': [t == T - 1 for t in range(T)],
        'reward': [np.array(0.0, dtype=np.float32)] * T,
        'discount': [np.array(1.0, dtype=np.float32)] * T,
    }

episodes = []
for _ in range(6):
    while True:
        state_obs, acs, state_gt, img_obs, dones = gen_one_traj_img(x_min, x_max, y_min, y_max, u_max, radius, dt, v, dpi, rand=-1)
        if len(dones) > 16 :
            break
    episodes.append(traj_to_episode(state_obs, acs, state_gt, img_obs, dones))

lengths = [len(ep['reward']) for ep in episodes]
max_T = min(max(lengths), 64)
if max_T < 6:
    raise ValueError('Not enough steps in episodes')

def extend_episode_with_dummy(ep, target_len):
    """Extend episode to target_len by appending dummy steps after done (same idea as dreamer_offline fixed-length segments)."""
    out = {}
    L = len(ep['reward'])
    for k, v in ep.items():
        arr = np.array(v)
        if L < target_len:
            n_dummy = target_len - L
            if k == 'image':
                dummy = np.zeros((n_dummy,) + arr.shape[1:], dtype=arr.dtype)
            elif k == 'action':
                dummy = np.tile(np.array([0, 1, 0], dtype=arr.dtype), (n_dummy, 1))
            elif k == 'state':
                dummy = np.zeros((n_dummy,) + arr.shape[1:], dtype=arr.dtype)
            elif k == 'obs_state':
                dummy = np.zeros((n_dummy,) + arr.shape[1:], dtype=arr.dtype)
            elif k == 'is_first':
                dummy = np.zeros(n_dummy, dtype=arr.dtype)
            elif k == 'is_terminal':
                dummy = np.ones(n_dummy, dtype=arr.dtype)
            elif k in ('reward', 'discount'):
                dummy = np.zeros(n_dummy, dtype=arr.dtype)
            else:
                dummy = np.repeat(arr[-1:], n_dummy, axis=0)
            arr = np.concatenate([arr, dummy], axis=0)
        out[k] = arr[:target_len]
    return out

extended = [extend_episode_with_dummy(ep, max_T) for ep in episodes]
batch = {k: np.stack([p[k] for p in extended], axis=0) for k in extended[0].keys()}
# lengths kept as actual trajectory lengths for temporal repeat in visualization
lengths = [min(L, max_T) for L in lengths]

def to_torch_batch(b):

    return {k: torch.tensor(v, device=config.device, dtype=torch.float32) if np.issubdtype(v.dtype, np.floating) or np.issubdtype(v.dtype, np.integer)
            else torch.tensor(v, device=config.device) for k, v in b.items()}
batch_t = to_torch_batch(batch)
batch_t['cont'] = (1.0 - batch_t['is_terminal'].float()).unsqueeze(-1)

## Visualize World Model Imagination

TODO: Run the following cell to visualize the world model's imagination.

- First row: Ground truth trajectory (starting from random initial states and rolling out actions until boundary state.)
- Second row: World model's imagination (decoded to images) using the initial observations only and with the same action sequences.
- Third row: Pixel-wise difference between the g.t. and the world model's imagination.

In [ ]:
#@title Roll out WM Imagination.

initial_wm_checkpoint = 'best_pretrain_joint_0_79.pt'
middle_wm_checkpoint = 'best_pretrain_joint_0_40.pt'
final_wm_checkpoint = 'pretrain_joint.pt'
checkpoint_paths = [('initial', initial_wm_checkpoint), ('middle', middle_wm_checkpoint), ('final', final_wm_checkpoint)]
out_dir = pathlib.Path('eais_hw2/dreamerv3-torch/wm_viz_videos')
out_dir.mkdir(parents=True, exist_ok=True)

for name, ckpt_path in checkpoint_paths:
    if not ckpt_path or not pathlib.Path(ckpt_path).exists():
        print(f'Skip {name}: path not set or missing')
        continue
    ckpt = torch.load(ckpt_path, map_location=config.device)
    state_dict = ckpt['agent_state_dict']
    # Strip torch.compile wrapper prefix if present (_wm._orig_mod.)
    prefix = '_wm._orig_mod.'
    state_dict = {k[len(prefix):] if k.startswith(prefix) else k: v for k, v in state_dict.items()}
    agent.load_state_dict(state_dict, strict=True)
    agent.eval()
    with torch.no_grad():
        vid = agent.video_pred(batch_t)
    vid = to_np(vid)
    for i in range(6):
        L = lengths[i]
        if L < vid.shape[1]:
            vid[i, L:] = vid[i, L-1:L]
    if np.issubdtype(vid.dtype, np.floating):
        vid = np.clip(255 * vid, 0, 255).astype(np.uint8)


    out_file = out_dir / f'wm_{name}.mp4'

    x = vid
    B, T, H, W, C = x.shape

    # If float, assume [0,1] or [-1,1] and convert to uint8
    if np.issubdtype(x.dtype, np.floating):
        if x.min() < 0:  # likely [-1,1]
            x = (x * 0.5 + 0.5)
        x = np.clip(x, 0, 1)
        x = (x * 255.0 + 0.5).astype(np.uint8)
    else:
        x = np.clip(x, 0, 255).astype(np.uint8)

    # If grayscale, make it 3-channel
    if C == 1:
        x = np.repeat(x, 3, axis=-1)

    # Build (T, H, B*W, C) by concatenating batches horizontally per timestep
    cat_frames = []
    for t in range(T):
        # x[:, t] is (B,H,W,C); concatenate along width
        frame_t = np.concatenate([x[b, t] for b in range(B)], axis=1)  # axis=1 is width in (H,W,C)
        cat_frames.append(frame_t)

    cat_frames = np.stack(cat_frames, axis=0)  # (T, H, B*W, C)

    # Write mp4 (macro_block_size=1 avoids forced resizing to multiples of 16)
    imageio.mimsave(
        str(out_file),
        cat_frames,
        fps=4,
        codec="libx264",
        quality=8,
        macro_block_size=1,
    )

# Visualize saved videos in the notebook
from IPython.display import Video, display
for name, _ in checkpoint_paths:
    p = out_dir / f'wm_{name}.mp4'
    if p.exists():
        print(f'--- {name} ---')
        display(Video(str(p), embed=True))

## Visualize Safety Margin Function

TODO: Run the following cell to visualize the pretrained safety margin function.

In [ ]:
#@title Visualize Safety Margin (May take some time)
import sys
import pathlib
import io
import os
import numpy as np
import torch
import gym
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ruamel.yaml as yaml
from PIL import Image
from torch import nn
from torch.nn.utils import spectral_norm

os.environ["MUJOCO_GL"] = "osmesa"

wm_root = pathlib.Path('eais_hw2/dreamerv3-torch').resolve()
if str(wm_root) not in sys.path:
    sys.path.insert(0, str(wm_root))
if str(wm_root.parent) not in sys.path:
    sys.path.insert(0, str(wm_root.parent))
import tools
import models

to_np = lambda x: x.detach().cpu().numpy()

# --- Load config ---
def recursive_update(base, update):
    for k, v in update.items():
        if isinstance(v, dict) and k in base:
            recursive_update(base[k], v)
        else:
            base[k] = v

config_path = pathlib.Path('eais_hw2/configs.yaml').resolve()
yaml_loader = yaml.YAML(typ='safe', pure=True)
configs = yaml_loader.load(config_path.read_text())
defaults = {}
recursive_update(defaults, configs['defaults'])

class Config:
    pass
config = Config()
for k, v in defaults.items():
    setattr(config, k, v)
config.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config.num_actions = 3

# --- Build observation / action spaces ---
bounds = np.array([[config.x_min, config.x_max], [config.y_min, config.y_max], [0, 2 * np.pi]])
low, high = bounds[:, 0], bounds[:, 1]
midpoint, interval = (low + high) / 2.0, high - low
image_size = config.size[0]
observation_space = gym.spaces.Dict({
    'state': gym.spaces.Box(np.float32(midpoint - interval/2), np.float32(midpoint + interval/2)),
    'obs_state': gym.spaces.Box(low=-1, high=1, shape=(2,), dtype=np.float32),
    'image': gym.spaces.Box(low=0, high=255, shape=(image_size, image_size, 3), dtype=np.uint8),
})
action_space = gym.spaces.Discrete(3)

# --- Build world model ---
wm = models.WorldModel(observation_space, action_space, 0, config).to(config.device)

# --- Load WM checkpoint ---
wm_ckpt_path = 'pretrain_joint.pt'
wm_ckpt = torch.load(wm_ckpt_path, map_location=config.device)
wm_state = wm_ckpt['agent_state_dict']
# Strip torch.compile prefix
prefix = '_wm._orig_mod.'
wm_state = {k[len(prefix):] if k.startswith(prefix) else k: v for k, v in wm_state.items()}
wm.load_state_dict(wm_state, strict=True)
wm.eval()
print(f"Loaded WM from {wm_ckpt_path}")

# --- Build and load classifier ---
feat_size = config.dyn_stoch + config.dyn_deter  # continuous latent
lx_mlp = nn.Sequential(
    spectral_norm(nn.Linear(feat_size, 16)),
    nn.ReLU(),
    spectral_norm(nn.Linear(16, 1)),
).to(config.device)

lx_ckpt_path = 'classifier.pt'
lx_ckpt = torch.load(lx_ckpt_path, map_location=config.device)
lx_mlp.load_state_dict(lx_ckpt['agent_state_dict'])
lx_mlp.eval()
print(f"Loaded classifier from {lx_ckpt_path}")

# --- Helper: render synthetic observation image ---
def capture_image(state):
    fig, ax = plt.subplots()
    plt.xlim([config.x_min, config.x_max])
    plt.ylim([config.y_min, config.y_max])
    plt.axis("off")
    fig.set_size_inches(1, 1)
    circle = patches.Circle([0, 0], config.obs_r, edgecolor=(1, 0, 0), facecolor="none")
    ax.add_patch(circle)
    dt, v = config.dt, config.speed
    plt.quiver(
        state[0], state[1],
        dt * v * np.cos(state[2]), dt * v * np.sin(state[2]),
        angles="xy", scale_units="xy", minlength=0,
        width=0.05, scale=0.2, color=(0, 0, 1), zorder=3,
    )
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=config.size[0])
    buf.seek(0)
    img = Image.open(buf).convert("RGB")
    img_array = np.array(img)
    plt.close()
    return img_array

# --- Helper: encode observations and run classifier ---
def get_latent(xs, ys, thetas, imgs):
    batch_size = xs.shape[0]
    states = np.expand_dims(np.expand_dims(thetas, 1), 1)
    imgs_arr = np.expand_dims(np.array(imgs), 1)
    dummy_acs = np.zeros((batch_size, 1, 3))
    dummy_acs[:, :, 1] = 1
    cos = np.cos(states)
    sin = np.sin(states)
    obs_state = np.concatenate([cos, sin], axis=-1)
    data = {
        "obs_state": obs_state,
        "image": imgs_arr,
        "action": dummy_acs,
        "is_first": np.ones((batch_size, 1)),
        "is_terminal": np.zeros((batch_size, 1)),
    }
    data = wm.preprocess(data)
    with torch.no_grad():
        embed = wm.encoder(data)
        post, _ = wm.dynamics.observe(embed, data["action"], data["is_first"])
        feat = wm.dynamics.get_feat(post)
        g_x = lx_mlp(feat).cpu().numpy().squeeze()
    return g_x

# --- Evaluate over grid ---
nx, ny, nz = 41, 41, 5
xs = np.linspace(-1, 1, nx)
ys = np.linspace(-1, 1, ny)
thetas = np.linspace(0, 2 * np.pi, nz, endpoint=True)

v_grid = np.zeros((nx, ny, nz))
idxs, imgs, labels = [], [], []
it = np.nditer(v_grid, flags=["multi_index"])

print("Rendering grid images...")
while not it.finished:
    idx = it.multi_index
    x, y, theta = xs[idx[0]], ys[idx[1]], thetas[idx[2]]
    is_unsafe = (x ** 2 + y ** 2) < (config.obs_r ** 2)
    labels.append(1 if is_unsafe else 0)
    x_offset = x - np.cos(theta) * 0.05
    y_offset = y - np.sin(theta) * 0.05
    imgs.append(capture_image(np.array([x_offset, y_offset, theta])))
    idxs.append(idx)
    it.iternext()

idxs = np.array(idxs)
labels = np.array(labels)
safe_idxs = np.where(labels == 0)
unsafe_idxs = np.where(labels == 1)

x_lin = xs[idxs[:, 0]]
y_lin = ys[idxs[:, 1]]
theta_lin = thetas[idxs[:, 2]]

# Forward pass in chunks
print("Running classifier on grid...")
g_x = []
num_chunks = 5
chunk_size = len(x_lin) // num_chunks
for k in range(num_chunks):
    s, e = k * chunk_size, (k + 1) * chunk_size
    g_xk = get_latent(x_lin[s:e], y_lin[s:e], theta_lin[s:e], imgs[s:e])
    g_x.extend(g_xk.tolist())
g_x = np.array(g_x)
v_grid[idxs[:, 0], idxs[:, 1], idxs[:, 2]] = g_x

# Confusion matrix
tp = np.where(g_x[safe_idxs] > 0)
fn = np.where(g_x[safe_idxs] <= 0)
fp = np.where(g_x[unsafe_idxs] > 0)
tn = np.where(g_x[unsafe_idxs] <= 0)

tp_n, fn_n = np.shape(tp)[1], np.shape(fn)[1]
fp_n, tn_n = np.shape(fp)[1], np.shape(tn)[1]
total = tp_n + fn_n + fp_n + tn_n

print(f"TP: {tp_n} ({tp_n/total*100:.1f}%), FN: {fn_n} ({fn_n/total*100:.1f}%), "
      f"TN: {tn_n} ({tn_n/total*100:.1f}%), FP: {fp_n} ({fp_n/total*100:.1f}%)")

# --- Plot ---
vmax = round(max(np.max(v_grid), 0), 1)
vmin = round(min(np.min(v_grid), -vmax), 1)
extent = [config.x_min, config.x_max, config.y_min, config.y_max]

fig, axes = plt.subplots(nz, 2, figsize=(6, nz * 2))
for i in range(nz):
    # Left: continuous g(x)
    ax = axes[i, 0]
    im = ax.imshow(
        v_grid[:, :, i].T, interpolation="none", extent=extent,
        origin="lower", cmap="seismic", vmin=vmin, vmax=vmax, zorder=-1,
    )
    cbar = fig.colorbar(im, ax=ax, pad=0.01, fraction=0.05, shrink=0.95, ticks=[vmin, 0, vmax])
    cbar.ax.set_yticklabels(labels=[vmin, 0, vmax], fontsize=24)
    ax.set_title(rf"$g(x)$, $\theta={thetas[i]:.2f}$", fontsize=18)
    circle = plt.Circle((0, 0), config.obs_r, fill=False, color="blue", label="GT boundary")
    ax.add_patch(circle)
    ax.set_aspect("equal")

    # Right: binary classification (g(x) > 0)
    ax = axes[i, 1]
    im = ax.imshow(
        v_grid[:, :, i].T > 0, interpolation="none", extent=extent,
        origin="lower", cmap="seismic", vmin=-1, vmax=1, zorder=-1,
    )
    cbar = fig.colorbar(im, ax=ax, pad=0.01, fraction=0.05, shrink=0.95, ticks=[vmin, 0, vmax])
    cbar.ax.set_yticklabels(labels=[vmin, 0, vmax], fontsize=24)
    ax.set_title(rf"$v(x)$, $\theta={thetas[i]:.2f}$", fontsize=18)
    circle = plt.Circle((0, 0), config.obs_r, fill=False, color="blue", label="GT boundary")
    ax.add_patch(circle)
    ax.set_aspect("equal")

fig.tight_layout()
fig.suptitle(
    rf"$TP={tp_n/total*100:.0f}\%$  $TN={tn_n/total*100:.0f}\%$  "
    rf"$FP={fp_n/total*100:.0f}\%$  $FN={fn_n/total*100:.0f}\%$",
    fontsize=14, y=1.01,
)
plt.show()

del lx_mlp, wm
torch.cuda.empty_cache()

---

### **Q3.1 Train Latent Safety Filter**

**TODO: Train the latent safety filter. Select a plotted checkpoint and comment on the quality of the learned value function (the ground truth, privileged-state value function is overlayed). Upload the plot to the markdown file.** ESTIMATED TIME: 1-2 hours

**This might take too long or lead to the OOM error in colab. There is a sample value function plot at `eais_hw2/safety_rl/latent_value_iter60000.png`. You may use this figure for the answer and proceed.**

Plot paths: logs/experiments/car-DDQN/latent-toEnd/figure/X0000.png

In [ ]:
!python eais_hw2/safety_rl/latent_dubins_avoid.py

### **Q3.2 List at least 2 reasons why you would expect a latent safety filter to return a lower-quality BRT than the methods that leverage privileged state (DeepReach or normal RL). What would make this BRT more accurate?**

**TODO: Your Markdown Solution Here**

### **Q3.3 Biased World Model**

Imagine that your world model was trained on a dataset that had *biased* coverage of the action space: you only ever saw the Dubins' car proceed straight or turn left ($u \in \{0, u_{max}\}$)  in the dataset, but you got to see this for all possible initial conditions of the robot. Imagine you had a perfect failure classifier $l(x)$. Given this biased WM and the perfect failure classifier and the perfect RL solver in the WM's imagination, what do you think the resulting BRT would look like? Would it be over- or under-conservative? Why?

**TODO: Your Markdown Solution Here**